<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.4.1. Polynomial Regression with SGD
  </strong>
</div>

### Background

You will be working with a synthetic dataset generated from a polynomial function with added noise. The dataset includes:

- 1000 data points
- 1 input feature (x)
- 1 target variable (y) generated by a polynomial function
- Random noise to simulate real-world data

### Description

Your task is to explore different approaches to fit this non-linear data using SGDRegressor. You should:

- Start by fitting a linear model and analyze its limitations
- Implement polynomial feature transformation
- Experiment with different learning rates and number of iterations
- Compare the performance of different polynomial degrees
- Visualize the learning process and final predictions

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **polynomial-regression-sgd.ipynb** that contains your code and analysis
- The **notebook** should include visualizations of the different models and their learning curves
- A discussion of the techniques you found most effective for minimizing the error
- Use the provided **polynomial-regression-sgd.ipynb** template as a starting point


In [1]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from rich.table import Table
from rich.console import Console

console = Console()

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_csv("./data/synthetic-polynomial-data.txt", sep="\t")
x = df["x"].values.reshape(-1, 1)
y = df["y"].values.reshape(-1, 1)

plt.style.use("ggplot")
plt.figure(figsize=(10, 6))
plt.scatter(x, y, color="m", marker="x", s=5)
plt.xlabel("x")
plt.ylabel("y")
plt.ylim(top=15000)
plt.xlim(right=55)
plt.show()

print(f"Shape of x: {x.shape}")
print(f"Shape of y: {y.shape}")

In [ ]:
linear_regressor = LinearRegression()
linear_regressor.fit(x, y)
y_hat = linear_regressor.predict(x)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, color="m", marker="x", s=5)
plt.plot(x, y_hat, color="g", linewidth=2)
plt.show()

print(f"Shape of y_hat: {y_hat.shape}")
print(f"Intercept: {linear_regressor.intercept_}")
print(f"Coefficients: {linear_regressor.coef_}")

In [ ]:
def my_mse_implementation(y: NDArray[np.float64], y_hat: NDArray[np.float64]) -> float:
    return np.mean((y - y_hat) ** 2)


mse_one = my_mse_implementation(y, y_hat)
mse_two = mean_squared_error(y, y_hat)

print(f"MSE one: {mse_one}")
print(f"MSE two: {mse_two}")

assert mse_one == mse_two

In [ ]:
linear_sgd_regressor = SGDRegressor()
linear_sgd_regressor.fit(x, y.ravel())
y_hat_sgd = linear_sgd_regressor.predict(x)
mse_three = mean_squared_error(y, y_hat_sgd)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, color="m", marker="x", s=5)
plt.plot(x, y_hat, color="g", linewidth=2)
plt.show()

print(f"Shape of y_hat_sgd: {y_hat_sgd.shape}")
print(f"MSE SGD: {mse_three}")
print(f"Intercept: {linear_sgd_regressor.intercept_}")
print(f"Coefficients: {linear_sgd_regressor.coef_}")

In [ ]:
comparison_linear_and_sgd = Table(title="Model Comparison")

comparison_linear_and_sgd.add_column("Model", justify="left", style="cyan")
comparison_linear_and_sgd.add_column("MSE", justify="right", style="magenta")
comparison_linear_and_sgd.add_column("R2", justify="right", style="magenta")
comparison_linear_and_sgd.add_column("Intercept", justify="right", style="green")
comparison_linear_and_sgd.add_column("Coefficient", justify="right", style="yellow")

comparison_linear_and_sgd.add_row(
    "Linear Regression",
    f"{mse_one:.4f}",
    f"{r2_score(y, y_hat):.4f}",
    f"{linear_regressor.intercept_[0]:.4f}",
    f"{linear_regressor.coef_[0][0]:.4f}",
)
comparison_linear_and_sgd.add_row(
    "SGD Regression",
    f"{mse_three:.4f}",
    f"{r2_score(y, y_hat_sgd):.4f}",
    f"{linear_sgd_regressor.intercept_[0]:.4f}",
    f"{linear_sgd_regressor.coef_[0]:.4f}",
)

console.print(comparison_linear_and_sgd)

In [ ]:
poly = PolynomialFeatures(degree=3, include_bias=False)
x_poly = poly.fit_transform(x)

scaler = StandardScaler()
x_poly_scaled = scaler.fit_transform(x_poly)

poly_sgd_regressor = SGDRegressor(learning_rate="constant", eta0=0.001)
poly_sgd_regressor.fit(x_poly_scaled, y.ravel())

# Create extended range for prediction
x_extended = np.linspace(-75, 75, 1000).reshape(-1, 1)
x_poly_extended = poly.transform(x_extended)
x_poly_extended_scaled = scaler.transform(x_poly_extended)
y_hat_poly_sgd_extended = poly_sgd_regressor.predict(x_poly_extended_scaled)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, color="m", marker="x", s=5, label="Training Data")
plt.plot(
    x_extended, y_hat_poly_sgd_extended, color="g", linewidth=2, label="Polynomial Fit"
)
plt.xlabel("x")
plt.ylabel("y")
plt.xlim(-75, 75)
plt.legend()
plt.show()

y_hat_poly_sgd = poly_sgd_regressor.predict(x_poly_scaled)
mse_poly_sgd = mean_squared_error(y, y_hat_poly_sgd)
r2_poly_sgd = r2_score(y, y_hat_poly_sgd)

In [ ]:
comparison_poly = Table(
    title="Polynomial Model Results",
    show_header=True,
    header_style="bold",
)

comparison_poly.add_column("Model")
comparison_poly.add_column("MSE")
comparison_poly.add_column("R2")
comparison_poly.add_column("Intercept")
comparison_poly.add_column("Coefficients")

comparison_poly.add_row(
    "Polynomial SGD",
    f"{mse_poly_sgd:.4f}",
    f"{r2_poly_sgd:.4f}",
    f"{poly_sgd_regressor.intercept_[0]:.4f}",
    str(poly_sgd_regressor.coef_),
)

console.print(comparison_poly)

# Analysis of Results

The polynomial regression model using Stochastic Gradient Descent (SGD) significantly outperformed the linear models, achieving:

- R² score of 0.9951 compared to ~0.79-0.81 for linear models

This improvement in performance indicates that the relationship between x and y is clearly non-linear and better modeled by a polynomial function. The polynomial SGD model was able to capture the underlying curved pattern in the data, as shown in the visualization where the green polynomial fit line closely follows the training data points.